# Creating the RAG with openAI

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")

### Creating the Model and Embeddings

In [2]:
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

model = ChatOpenAI(
    model = "gpt-5.4-nano",
    api_key = openai_api_key
)

embedding = OpenAIEmbeddings(
    model = "text-embedding-3-small",
    api_key = openai_api_key
)

### splitting the document

In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

file = "policy.pdf"
loader = PyPDFLoader(file)

docs = loader.load()

# splitting the documents
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 600,
    chunk_overlap = 120,
    add_start_index = True
)

all_splits = text_splitter.split_documents(docs)
print(f"Split policy into {len(all_splits)} sub-documents.")

: 

### creating a vector store

In [ ]:
from langchain_chroma import Chroma

# creating vector store
vector_store = Chroma(
    collection_name = "policy_collection",
    embedding_function = embedding,
    persist_directory = "./policy_db"
)

# storing the documents 
document_ids = vector_store.add_documents(
    documents = all_splits
)

: 

: 

### Retrieving the tool

In [6]:
from langchain.tools import tool

@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve information to help answer a query."""
    retrieved_docs = vector_store.similarity_search(query, k=2)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\nContent: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs

In [58]:
from langchain.agents import create_agent
from pydantic import BaseModel, Field
from typing import List

tools = [retrieve_context]

system_prompt = (
    "You are a Corporate Compliance Agent. "
    "1. If the user provides a receipt, audit it for violations (e.g., alcohol, price limits). "
    "2. If the user asks a general question, use retrieve_context to explain the policy rules. "
    "If the question is not about this policy document, ignore it"
)

class PolicyNodeResult(BaseModel):
    result: str

rag_agent = create_agent(
    model = model,
    tools = tools,
    system_prompt = system_prompt,
    response_format = PolicyNodeResult
)

In [60]:
from langchain.messages import HumanMessage

ques = "How much can I spend maximum on air travel on international trips for the company as an L1 employee?"
query = HumanMessage(content = ques)
response = rag_agent.invoke({
        "messages": [query]
})

print(response["structured_response"])



result='For international air travel, the policy excerpt retrieved does not show a specific maximum airfare cap for L1 employees. It only provides international communication limits for L1–L4 (call: INR 1,500 per trip; data roaming: INR 1,000 per trip) and a domestic expense limit table. If you want, I can help look for the airfare section or interpret the international travel allowance rules if you provide the relevant policy page.'


In [57]:
print(response["messages"])

[HumanMessage(content='How much can I spend maximum on air travel on international trips for the company?', additional_kwargs={}, response_metadata={}, id='c79431a3-3e2c-414a-86ff-91e66ade6d0e'), AIMessage(content='', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 370, 'total_tokens': 405, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-DQwa8oJZCCzencg5y1WlPtxsyNmFq', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d58f7-2840-73b0-979c-559453d5a678-0', tool_calls=[{'name': 'retrieve_context', 'args': {'query': 'company policy maximum spend for air travel on international trips; include relevant 